# Figure 1: Automated SWOT PIXC filtering across Chilean river reaches

**Main message:** The method was applied across a broad national-scale river domain and is designed as a repeatable workflow.

This notebook builds a publication figure with:

- **a.** Chile SWORD study reaches, colored by median absolute filtered PIXC slope on a logarithmic scale.
- **b.** The automated PIXC-to-slope processing workflow.
- **c.** A compact distribution of observations per reach.

The study set is defined from the full 2024 updated-filter comparison table: 7,214 processed observations across 347 SWORD reaches.

In [ ]:
from pathlib import Path
import base64
import zlib
from shapely.wkb import loads as load_wkb
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import pandas as pd

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "figure.dpi": 140,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

## Configuration

Panel A uses median absolute filtered PIXC slope magnitude to show the geomorphic range of the study domain. Absolute magnitude avoids conflating profile-direction sign conventions with river steepness.

In [ ]:
def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "raqw").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the SWOT_universal_PIXC_quantile_filter project root.")


PROJECT_ROOT = find_project_root()
FIGURE_DIR = PROJECT_ROOT / "publication_figs" / "outputs"

SWORD_GPKG = PROJECT_ROOT / "data" / "Petrohue_SWORD_reaches" / "chile_reaches.gpkg"
CHILE_OUTLINE_ZLIB_B64 = "..."
PERU_OUTLINE_ZLIB_B64 = "eNpdVGtQlFUYZnUTgo0hZvKSDWIFbrr77fk2ECwTWA0VdsLzuQoCyoKBN1BCRIV1AxUHZ9y4tKUQoqCwJJQoF/GCPiOJgiSLHwKWDpccATXHkEjFSweZ6Udn5syZd57nfZ/nfc+ZIxlvY2MjYVvL9maP4cKiTyiM7a4X6mUKaGaZXJZ6UfQfCVrWH6lAQPjlwBJfFqe4pN/doUDDoiCd3ofCmh1/2PGiAk7D1cm7Aiji/GMPfyNRwjGXtPSx2O7nF5eqZyhxcajsiDSIwvX08YVSdyUe1DlYXSlF+tyC+JBUJdzj7B7XBFNMnhkTE5mnRHOT9PsDIRRZ3JeNx4aViI2SbE9eTHEgOyti61YOyW4zthUJjJ/04GFhJYcnpvAMLYu7rl7OyWnh4K+32/xsGUX8q2bVul4Os6sSktLDKDqDG5stviqUF/h1twUyf/oS0588QersXzaeXUjxqE7wJp8SnMP+6pvzKBR/NRyv9yRw9NP+E+xJkf/RlKteOoLBVb2VV/woFk24ptUkElhPdYZnayjMU55+3mYmeLay+McMNi/pnhUJK4oIcjKa1kYw3LXDNvi7MoLlT+ytZUw/o3bQIfMMgWCUzNkWSlGr9lxS+QYPx96p08OYX2hMxXMn8thZ2Pv49zUUbb3OXJqKx8MyvsWynuKzhm+7fl3MY/b2KQsSIpm/rtoC4yoeE3uWbzqqp3B6XFB/PI3HzMpKx0sbKBquOEoGLTx0H8skYYkUu301HnWlPNQmn6rVSRSx9+7Yyk7weNdSVDfwFYXs2NPqDcM8jO/3+b3YQiHf6brlyXtqcCXJPV5fs/5ufOBsma/Gxt03HhlNFPuutJNmnRr70r9o0eSy+bu9vTdpuRrQG0SfWopbyQ4+/lI1ngdcl+89w/TDqgaWDPIoqTGIO0QKXV6U+WwVj/zb7sa8Tva+TDKfkpM8rl82iOM7KLzTagacingcY/yDN1m/iXpzSgyPPMb/rYvd1yiu5XGe6U0bpgjfWD1QoeARNWIQq59T6Jme1Zn1e88gZtoKkJlPDTyS8q/9HZIJsEZGmH+4RzD0yt3Y4CzgQ4bH9xD0/9EqL54sQLrW3ufueYLSUX03AXPvR5ivlhJcM7TKJWoBpSx/aiHBKqbnGSiAjOLrCBJDW+V/Cyw/hfUfTZAvvy6/rRPg9FJvLtON6d0NFTDkFGV2CGTvIfXOW52xAsotnecOziOQrZlhtN/E8kf5hGCeb3SzJlWAblzmacObBOVPU8SRnQJS2y/I999XYc+JrFuSbAHdWa4VNidVeOneNmF7roAlk1r3h1pUsAuaNlB0UMC26FlT30lUwZ7VD2gSkHVrvRjZyCEq2EU5p5n5qz2qjT7NQZhVED/YKCDENOH+yhMcPJQG8SfGzxjlH+KQVGwQO6wMrz8v90rlxvxcY/Ur6uQdcWP4pE4235q1YuECDpN3GcQN3QK8x8WJ09uVIKP3McJwj0vyrVolbF6vpcgIWi2b2af4Lx47Ffj/f/UvzY9bIA=="
BOLIVIA_OUTLINE_ZLIB_B64 = "eNplVGlQk1cUTQQ1I5pRq1OwiBTBBZPwJfnyKUtRIgoGlO+xQ1GWaqQWl4miiEIpIoOKC9FSWzWiFgRhmhakLohzRIoIqGjB0uqAk1YgxVBEoMCg9rX9176ZO2/OvDv3nnPunSe0EggEQhpraGg0DxNGM3ksUdhECHtUKNNcf3dyNo9n5rcmj1YVjnWErYk5ymM9v7X6SYMK3mT0pvE4j6cb33x67R0Oj6ZF99gaecQdeWQ+7c4hcra037ucR239ocKUzRwGtrAS2xoe3ck5Ik02h+C5dr6Xb/EoLnnOGi9wKD8ktiQ28ZhRUuWZ9oiDmoiUu7t4MLkxZeXPOfRuv78i9iUP3835ezN+5ZDNtNnXDfPI3VXYl/+Qw8nkQvPYEI+B8hJX+1wO/uNcW3vGKL9DpXeywzj0VUX1pL6meuxDWsNncShYfUq/YAJB7pTaZG5AhYSE9scminn7hU+dqb5x6ky/iskEZZkRug16FWRii8Z/CkEL5d/gr8IN2+Qfv55FkBW4vLZykEX7c8FI9lyCyHt2iUNmFnXua3OKnQlE1UsPSm6y0DUeENY7ENw4UmXoPctiwkj+YMciAsfvctVBWSx6Z+XtGWMIbA94WFo/YRHu19eV9jc2Wh76rGWheWbTnETzRU2zx78fysLZMvT+HhnB1brB4sJ5LIpur2m5piDIsdJ2BrzHQuttH3/RhaC4Vd/hLmbhgTiXEieCX45vjzb0KKG4dcbNTPk6Ok+TubQp8XvgqyyhlGBZAX9KekWJ0+P6ugrtCeq9Go3LtyqR4jSzy9mDYIm/tvOjaUpoE0Zvhi0lODGnynN1vwKKx8f2/OZFELdv4VNRuwKdpjnprmqCHQvWlamLFBiOHIpXUtwWflCkS1Xgknrm/KSVtL9VmnzUU4GxdlFF/yqClAAkd85VgG8wWzxWEKz/oyDEa5ICel2uITqEQGCYLjOPyFG0aU66IIKAzbK+rq2RozjgzZPmWIJQ8dmQGWfl4Hbb+eriCd3bq4b0dXL0lsofXNxEwNQd77i3So7Mc6ZXTzYSNJmmyz5zk6P+7hm3sSgCL6tKoeNUOYLThe4p0QS1ChVfMV6O8OFJzaUBVO/Vfpuj1xnocxoSY6keyU8TIz4vZTC6trAkZxnB1OwoXdR5Bs1X2mLy6HuB3ciKlhMM+teZKup9aP0J9wPVOxiIfQL/jFDRegvtGheHMriBLyp/9qb+vqwz3lYxyOBqt1T5EXRXBy9hPBmUGXyetdD+Y7FFhy1yBtxl3c79H9L5RNxturjMDSvjRNtHw+j83za5fWySYfhwTE5gMEFz4x29/oEMe13mp5ynWLDzRe+5ChmSEoS7U6nf3+Ydi921S4amBuv8k5F0X2Tr714akmLBZtGr76m/IletNv4rKV5U2zQ7EoJtXoZtkRlS1AyUXrAOonyvGf2s50kh/pJ50KWh8/zm9Q+V86WYOlSZuo/ixJVJBUeEUtT5B4XG0X3pzttWIK6RQBNzJ6CI+iXY67C/M00C9aLDDiGLKd8LQWHd8RLM882cOOxOcUZllcMMCeycnCY6fEDz/zmSf+8M/n94w3i/4MEXKvz3P/sLGJNaIA=="
ANALYSIS_OBSERVATIONS_CSV = (
    PROJECT_ROOT
    / "data"
    / "riversp_comparison_updated_filter"
    / "matched_pixc_riversp_slopes_with_residuals.csv"
)

COLOR_BY = "median_filtered_slope_magnitude"
OUTPUT_STEM = "figure_01_study_domain_and_workflow"
EXPORT_FILES = False  # Set True only when PNG/PDF files are needed.
if EXPORT_FILES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

assert SWORD_GPKG.exists(), SWORD_GPKG
assert ANALYSIS_OBSERVATIONS_CSV.exists(), ANALYSIS_OBSERVATIONS_CSV
PROJECT_ROOT

## Assemble the reach-level figure table

Each row of the saved 2024 comparison table is one processed reach-overpass observation. This is the full national analysis population used for the figure, not the smaller later RAQW validation subset.

In [ ]:
observations = pd.read_csv(ANALYSIS_OBSERVATIONS_CSV, dtype={"reach_id": str})
observations["reach_id"] = observations["reach_id"].astype(str)
observations["observation_date"] = pd.to_datetime(observations["obs_date_utc"], errors="coerce")
observations["abs_filtered_slope_m_per_km"] = observations["updated_postfilter_slope_m_per_km"].abs()

required_columns = {
    "reach_id", "year", "file", "obs_date_utc",
    "updated_postfilter_slope_m_per_km", "riversp_slope_m_per_km",
}
missing_columns = required_columns - set(observations.columns)
if missing_columns:
    raise ValueError(f"Analysis table is missing required columns: {sorted(missing_columns)}")

duplicate_profiles = observations.duplicated(["reach_id", "file"]).sum()
if duplicate_profiles:
    raise ValueError(f"Found {duplicate_profiles} duplicate reach/file observations.")

reach_summary = (
    observations
    .groupby("reach_id", as_index=False)
    .agg(
        n_valid_observations=("file", "size"),
        n_unique_reach_dates=("observation_date", "nunique"),
        first_observation=("observation_date", "min"),
        last_observation=("observation_date", "max"),
        median_filtered_slope_m_per_km=("updated_postfilter_slope_m_per_km", "median"),
        median_abs_filtered_slope_m_per_km=("abs_filtered_slope_m_per_km", "median"),
        median_riversp_slope_m_per_km=("riversp_slope_m_per_km", "median"),
    )
)

reach_summary.sort_values("n_valid_observations", ascending=False).head()

In [ ]:
sword = gpd.read_file(SWORD_GPKG)
sword["reach_id"] = sword["reach_id"].astype("int64").astype(str)
sword = sword.to_crs(4326)
chile_outline_geometry = load_wkb(zlib.decompress(base64.b64decode(CHILE_OUTLINE_ZLIB_B64)))
chile_outline = gpd.GeoDataFrame(geometry=[chile_outline_geometry], crs=4326)
peru_outline_geometry = load_wkb(zlib.decompress(base64.b64decode(PERU_OUTLINE_ZLIB_B64)))
bolivia_outline_geometry = load_wkb(zlib.decompress(base64.b64decode(BOLIVIA_OUTLINE_ZLIB_B64)))
neighbor_outlines = gpd.GeoDataFrame(
    {"country": ["Peru", "Bolivia"]},
    geometry=[peru_outline_geometry, bolivia_outline_geometry],
    crs=4326,
)

study_reaches = sword.merge(reach_summary, on="reach_id", how="inner")
missing_geometry = sorted(set(reach_summary["reach_id"]) - set(study_reaches["reach_id"]))
if missing_geometry:
    raise ValueError(f"Analysis reaches missing from the SWORD GeoPackage: {missing_geometry}")

print(f"Chile SWORD reaches available: {len(sword):,}")
print(f"Study reaches in the full 2024 analysis: {len(study_reaches):,}")
print(f"Valid reach-overpass observations: {reach_summary['n_valid_observations'].sum():,}")
print(f"Unique reach-date combinations: {reach_summary['n_unique_reach_dates'].sum():,}")
print(
    "Observation period: "
    f"{reach_summary['first_observation'].min():%Y-%m-%d} to "
    f"{reach_summary['last_observation'].max():%Y-%m-%d}"
)
reach_summary["n_valid_observations"].describe()

## Figure functions

In [ ]:
from matplotlib.ticker import MaxNLocator

INK = "#17222B"
MUTED = "#5C6870"
GRID = "#DDE3E6"
BLUE = "#245A73"
TEAL = "#218C8D"
RED = "#C94C5C"


def build_figure(sword, study_reaches, reach_summary, chile_outline, neighbor_outlines, color_by):
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.labelsize": 9,
        "axes.labelcolor": INK,
        "axes.edgecolor": INK,
        "axes.linewidth": 0.8,
        "xtick.color": INK,
        "ytick.color": INK,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "figure.dpi": 160,
        "savefig.dpi": 600,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    def panel_heading(ax, letter, title):
        heading_x = 0.035 if letter in {"b", "c"} else 0
        ax.text(
            heading_x, 1.025, f"{letter}   {title}", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=10.5,
            fontweight="bold", color=INK,
        )

    def draw_map(ax):
        settings = COLOR_SETTINGS[color_by]
        column = settings["column"]
        values = study_reaches[column].dropna()
        vmin, vmax = np.nanpercentile(values, [2, 98])
        if np.isclose(vmin, vmax):
            vmin, vmax = float(values.min()), float(values.max()) + 1e-9
        norm = LogNorm(vmin=max(vmin, 0.01), vmax=vmax, clip=True) if settings.get("log") else Normalize(vmin=vmin, vmax=vmax, clip=True)

        ax.set_facecolor("#FAFBFB")
        neighbor_outlines.plot(ax=ax, facecolor="#F7F8F8", edgecolor="#8A969C", linewidth=0.65, zorder=0)
        chile_outline.plot(ax=ax, facecolor="#F4F6F7", edgecolor="#65747C", linewidth=0.75, zorder=0)
        sword.plot(ax=ax, color="#C3CCD1", linewidth=0.45, alpha=0.95, zorder=1)
        study_reaches.plot(
            ax=ax, column=column, cmap=settings["cmap"], norm=norm,
            linewidth=1.7, missing_kwds={"color": "#8D969B"}, zorder=3,
        )
        minx, miny, maxx, maxy = sword.total_bounds
        ax.set_xlim(minx - 0.35, maxx + 0.35)
        ax.set_ylim(miny - 0.45, maxy + 0.45)
        ax.set_aspect(1 / np.cos(np.deg2rad(0.5 * (miny + maxy))))
        ax.set_anchor("E")
        ax.xaxis.set_major_locator(MaxNLocator(3))
        ax.yaxis.set_major_locator(MaxNLocator(7))
        ax.grid(color=GRID, linewidth=0.35, alpha=0.7, zorder=0)
        ax.set_axisbelow(True)
        ax.set_xlabel("Longitude", labelpad=28)
        ax.set_ylabel("Latitude")
        ax.spines[["top", "right"]].set_visible(False)
        panel_heading(ax, "a", "RiverSP Reaches")

        scalar = mpl.cm.ScalarMappable(norm=norm, cmap=settings["cmap"])
        scalar.set_array([])
        cax = ax.inset_axes([0.12, -0.055, 0.76, 0.018])
        cbar = ax.figure.colorbar(scalar, cax=cax, orientation="horizontal")
        cbar.outline.set_linewidth(0.6)
        cbar.ax.tick_params(labelsize=7, length=2, pad=1)
        cbar.set_label(settings["label"], fontsize=7.2, labelpad=2)
        if settings.get("log"):
            cbar.set_ticks([0.1, 1, 10])
            cbar.set_ticklabels(["0.1", "1", "10"])

        country_labels = [
            ("PERU", -71.7, -16.0),
            ("BOLIVIA", -68.2, -16.1),
            ("CHILE", -71.0, -26.8),
        ]
        for label, x, y in country_labels:
            ax.text(
                x, y, label, ha="center", va="center", fontsize=6.8,
                fontweight="bold", color="#69767C", alpha=0.9, zorder=2,
                path_effects=[pe.withStroke(linewidth=2, foreground="white")],
            )

    def workflow_box(ax, center, text, number, facecolor, width=0.205, height=0.19):
        x, y = center
        x0, y0 = x - width / 2, y - height / 2
        ax.add_patch(FancyBboxPatch(
            (x0, y0), width, height,
            boxstyle="round,pad=0.012,rounding_size=0.018",
            transform=ax.transAxes, facecolor=facecolor,
            edgecolor="#8EA0A9", linewidth=0.9, zorder=2,
        ))
        ax.text(x0 + 0.014, y0 + height - 0.014, str(number),
                transform=ax.transAxes, ha="center", va="center",
                fontsize=6.5, fontweight="bold", color="white", zorder=5,
                bbox={"boxstyle": "circle,pad=0.18", "facecolor": BLUE,
                      "edgecolor": "white", "linewidth": 0.6})
        ax.text(x, y - 0.006, text, transform=ax.transAxes,
                ha="center", va="center", fontsize=7.9,
                color=INK, linespacing=1.16, zorder=4)

    def workflow_arrow(ax, start, end):
        ax.add_patch(FancyArrowPatch(
            start, end, transform=ax.transAxes, arrowstyle="-|>",
            mutation_scale=9, linewidth=1.0, color=MUTED, zorder=1,
        ))

    def draw_workflow(ax):
        ax.set_axis_off()
        panel_heading(ax, "b", "Automated processing workflow")
        top_y, bottom_y = 0.61, 0.25
        top_x = [0.115, 0.37, 0.625, 0.88]
        bottom_x = [0.88, 0.59, 0.30]
        width, height = 0.205, 0.17
        positions = [(x, top_y) for x in top_x] + [(x, bottom_y) for x in bottom_x]
        labels = [
            "Inputs\nPIXC, RiverSP, SWORD", "Reach assignment +\nspatial filtering",
            "Reach-year\nquantile-slope curves", "Stable quantile-\nrange selection",
            "Filtered PIXC\nprofile", "Reach-date\nOLS slope",
            "Compare with RiverSP\n$slope$ and $slope2$",
        ]
        colors = ["#EDF2F4"] * 7
        for number, (position, label, color) in enumerate(zip(positions, labels, colors), start=1):
            workflow_box(ax, position, label, number, color, width, height)

        half_w, half_h = width / 2, height / 2
        for left, right in zip(top_x[:-1], top_x[1:]):
            workflow_arrow(ax, (left + half_w + 0.010, top_y), (right - half_w - 0.010, top_y))
        workflow_arrow(ax, (top_x[-1], top_y - half_h - 0.010),
                       (bottom_x[0], bottom_y + half_h + 0.010))
        for right, left in zip(bottom_x[:-1], bottom_x[1:]):
            workflow_arrow(ax, (right - half_w - 0.010, bottom_y),
                           (left + half_w + 0.010, bottom_y))
        ax.text(
            0.5, 0.01,
            "The workflow produces one filtered PIXC slope estimate for each valid reach-overpass.",
            transform=ax.transAxes, ha="center", va="bottom", fontsize=8, color=MUTED,
        )

    def draw_histogram(ax):
        values = reach_summary["n_valid_observations"]
        lower = int(np.floor(values.min() / 5) * 5)
        upper = int(np.ceil(values.max() / 5) * 5)
        bins = np.arange(lower, upper + 5, 5)
        ax.hist(values, bins=bins, color=TEAL, edgecolor="white", linewidth=0.8, alpha=0.95)
        median = values.median()
        ax.axvline(median, color=RED, linestyle=(0, (4, 2)), linewidth=1.35)
        ax.text(median, ax.get_ylim()[1] * 0.92, f"Median = {median:.0f}",
                color=RED, fontsize=8, ha="center", va="top",
                bbox={"facecolor": "white", "edgecolor": "none", "pad": 1.5})
        ax.text(0.965, 0.89,
                f"{values.sum():,} total reach-overpasses\nn = {len(values):,} reaches",
                transform=ax.transAxes,
                ha="right", va="top", fontsize=8, color=MUTED)
        ax.set_xlim(lower, upper)
        ax.set_xlabel("Valid reach-overpasses per reach")
        ax.set_ylabel("Number of reaches")
        ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))
        ax.grid(axis="y", color=GRID, linewidth=0.55)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
        panel_heading(ax, "c", "Valid reach-overpasses per reach")

    fig = plt.figure(figsize=(10.6, 6.25), facecolor="white")
    grid = fig.add_gridspec(
        2, 2, width_ratios=[0.78, 1.72], height_ratios=[1.02, 0.98],
        left=0.055, right=0.99, bottom=0.115, top=0.965,
        wspace=0.105, hspace=0.10,
    )
    draw_map(fig.add_subplot(grid[:, 0]))
    draw_workflow(fig.add_subplot(grid[0, 1]))
    draw_histogram(fig.add_subplot(grid[1, 1]))
    return fig


COLOR_SETTINGS = {
    "median_filtered_slope_magnitude": {
        "column": "median_abs_filtered_slope_m_per_km",
        "label": "Median filtered slope (m km$^{-1}$)",
        "cmap": "viridis",
        "log": True,
    },
    "median_riversp_slope": {
        "column": "median_riversp_slope_m_per_km",
        "label": "Median RiverSP slope (m km$^{-1}$)",
        "cmap": "magma_r",
    },
}


def add_panel_label(ax, label):
    ax.text(
        0.01, 0.99, label,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=12, fontweight="bold",
        bbox={"facecolor": "white", "edgecolor": "none", "pad": 1.5, "alpha": 0.9},
        zorder=20,
    )


def plot_study_map(ax, all_reaches, selected_reaches, color_by):
    settings = COLOR_SETTINGS[color_by]
    column = settings["column"]
    values = selected_reaches[column].dropna()
    if values.empty:
        raise ValueError(f"No finite values are available for {column!r}.")

    # Percentile clipping prevents one extreme reach from flattening the color range.
    vmin, vmax = np.nanpercentile(values, [2, 98])
    if np.isclose(vmin, vmax):
        vmin, vmax = float(values.min()), float(values.max()) + 1e-9
    norm = Normalize(vmin=vmin, vmax=vmax, clip=True)

    all_reaches.plot(ax=ax, color="#D8D8D8", linewidth=0.28, alpha=0.8, zorder=1)
    selected_reaches.plot(
        ax=ax,
        column=column,
        cmap=settings["cmap"],
        norm=norm,
        linewidth=1.8,
        missing_kwds={"color": "#8C8C8C", "label": "No RiverSP match"},
        zorder=3,
    )

    minx, miny, maxx, maxy = all_reaches.total_bounds
    xpad = (maxx - minx) * 0.05
    ypad = (maxy - miny) * 0.02
    ax.set_xlim(minx - xpad, maxx + xpad)
    ax.set_ylim(miny - ypad, maxy + ypad)
    mid_lat = 0.5 * (miny + maxy)
    ax.set_aspect(1 / np.cos(np.deg2rad(mid_lat)))
    ax.set_xlabel("Longitude (degrees east)")
    ax.set_ylabel("Latitude (degrees north)")
    ax.set_title("Chile study reaches", pad=8, fontweight="bold")
    ax.grid(color="#E6E6E6", linewidth=0.5, zorder=0)

    scalar = mpl.cm.ScalarMappable(norm=norm, cmap=settings["cmap"])
    scalar.set_array([])
    cbar = ax.figure.colorbar(scalar, ax=ax, orientation="horizontal", pad=0.055, fraction=0.045)
    cbar.set_label(settings["label"])
    if color_by == "observation_count":
        cbar.ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True, nbins=5))

    ax.text(
        0.03, 0.03,
        f"{len(selected_reaches):,} reaches | "
        f"{selected_reaches['n_valid_observations'].sum():,} observations",
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=8,
        bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#B8B8B8", "alpha": 0.95},
        zorder=10,
    )
    add_panel_label(ax, "a")


def workflow_box(ax, center, text, width=0.25, height=0.13, facecolor="#EAF2F8"):
    x, y = center
    patch = FancyBboxPatch(
        (x - width / 2, y - height / 2), width, height,
        boxstyle="round,pad=0.012,rounding_size=0.018",
        transform=ax.transAxes,
        facecolor=facecolor,
        edgecolor="#315A72",
        linewidth=1.0,
    )
    ax.add_patch(patch)
    ax.text(x, y, text, transform=ax.transAxes, ha="center", va="center", fontsize=8)
    return patch


def workflow_arrow(ax, start, end):
    arrow = FancyArrowPatch(
        start, end,
        transform=ax.transAxes,
        arrowstyle="-|>",
        mutation_scale=10,
        linewidth=1.0,
        color="#4A4A4A",
        shrinkA=7,
        shrinkB=7,
    )
    ax.add_patch(arrow)


def plot_workflow(ax):
    ax.set_axis_off()
    ax.set_title("Automated processing workflow", pad=8, fontweight="bold")

    positions = [
        (0.16, 0.77), (0.50, 0.77), (0.84, 0.77),
        (0.84, 0.48), (0.50, 0.48), (0.16, 0.48),
        (0.50, 0.19),
    ]
    labels = [
        "SWOT PIXC\n+ RiverSP + SWORD",
        "Geospatial\nfiltering",
        "Reach-year quantile\ncharacterization",
        r"$\tau$ range\nselection",
        "Retained PIXC\nprofile",
        "OLS slope\nestimate",
        "Compare with RiverSP\nslope and slope2",
    ]
    colors = ["#DDEBF7", "#EAF2F8", "#EAF2F8", "#FFF2CC", "#E2F0D9", "#E2F0D9", "#FCE4D6"]
    for position, label, color in zip(positions, labels, colors):
        workflow_box(ax, position, label, facecolor=color)
    for start, end in zip(positions[:-1], positions[1:]):
        workflow_arrow(ax, start, end)

    ax.text(
        0.5, 0.02,
        "Repeatable by reach and year; each processed overpass yields a filtered slope estimate.",
        transform=ax.transAxes,
        ha="center", va="bottom", fontsize=8, color="#555555",
    )
    add_panel_label(ax, "b")


def plot_observation_histogram(ax, summary):
    values = summary["n_valid_observations"]
    bins = np.arange(0, values.max() + 10, 10)
    if len(bins) < 3:
        bins = 8
    ax.hist(values, bins=bins, color="#2A788E", edgecolor="white", linewidth=0.8)
    ax.axvline(values.median(), color="#D1495B", linestyle="--", linewidth=1.2, label=f"Median = {values.median():.0f}")
    ax.set_title("Observations per reach", pad=8, fontweight="bold")
    ax.set_xlabel("Valid reach-overpass observations")
    ax.set_ylabel("Number of reaches")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    add_panel_label(ax, "c")

## Build and export Figure 1

In [ ]:
fig = build_figure(sword, study_reaches, reach_summary, chile_outline, neighbor_outlines, COLOR_BY)

if EXPORT_FILES:
    png_path = FIGURE_DIR / f"{OUTPUT_STEM}_{COLOR_BY}.png"
    pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}_{COLOR_BY}.pdf"
    fig.savefig(png_path, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    print(f"Saved {png_path}")
    print(f"Saved {pdf_path}")
plt.show()



## Draft caption

**Figure 1. Study domain and automated SWOT PIXC filtering workflow.** (a) The 347 SWORD reaches included in the 2024 analysis, colored by median absolute filtered PIXC-derived slope magnitude on a logarithmic scale; the broader SWORD network is shown in gray. Chile, Peru, and Bolivia boundaries are derived from the Natural Earth 1:10 million admin-0 dataset. (b) Processing workflow used to generate one filtered PIXC slope estimate for each valid reach-overpass, followed by comparison with RiverSP slope and slope2. (c) Distribution of 7,214 valid reach-overpasses among the 347 study reaches.